In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

behavior_columns = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions"
]

news_columns = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities"
]

behaviors = pd.read_csv(
    "C:\\Users\\admin\\ML_RESUME_PROJECT\\behaviors.tsv",
    sep="\t",
    names=behavior_columns
)

news = pd.read_csv(
    "C:\\Users\\admin\\ML_RESUME_PROJECT\\news.tsv",
    sep="\t",
    names=news_columns
)

print("Behaviors:", behaviors.shape)
print("News:", news.shape)

Behaviors: (156965, 5)
News: (51282, 8)


In [2]:
def count_impressions(x):
    if pd.isna(x):
        return 0
    return len(x.split())


def count_clicks(x):
    if pd.isna(x):
        return 0
    return sum(item.endswith("-1") for item in x.split())


behaviors["impression_count"] = (
    behaviors["impressions"].apply(count_impressions)
)

behaviors["click_count"] = (
    behaviors["impressions"].apply(count_clicks)
)

behaviors["history_length"] = (
    behaviors["history"]
    .fillna("")
    .apply(lambda x: len(x.split()) if x else 0)
)

In [3]:
user_features = (
    behaviors
    .groupby("user_id")
    .agg(
        session_count=("impression_id", "count"),
        total_impressions=("impression_count", "sum"),
        total_clicks=("click_count", "sum"),
        history_length=("history_length", "max")
    )
    .reset_index()
)

user_features["overall_ctr"] = (
    user_features["total_clicks"] /
    user_features["total_impressions"]
)

user_features.head()

,user_id,session_count,total_impressions,total_clicks,history_length,overall_ctr
0,U100,1,105,1,10,0.009524
1,U1000,3,205,4,3,0.019512
2,U10001,2,84,3,15,0.035714
3,U10003,2,106,3,8,0.028302
4,U10008,1,72,1,23,0.013889


In [4]:
print("Number of user profiles:", len(user_features))
print()

print(
    user_features[
        [
            "session_count",
            "total_impressions",
            "total_clicks",
            "overall_ctr",
            "history_length"
        ]
    ].describe()
)

Number of user profiles: 50000

       session_count  total_impressions  total_clicks   overall_ctr  \
count   50000.000000       50000.000000  50000.000000  50000.000000   
mean        3.139300         116.868880      4.726880      0.085461   
std         3.001026         145.809655      5.768136      0.104511   
min         1.000000           2.000000      1.000000      0.003448   
25%         1.000000          25.000000      1.000000      0.028986   
50%         2.000000          67.000000      3.000000      0.048544   
75%         4.000000         152.000000      6.000000      0.090909   
max        62.000000        1960.000000    129.000000      0.666667   

       history_length  
count    50000.000000  
mean        18.521160  
std         23.900679  
min          0.000000  
25%          5.000000  
50%         11.000000  
75%         22.000000  
max        558.000000  


In [5]:
news_to_category = dict(
    zip(news["news_id"], news["category"])
)

news[["news_id", "category"]].head(10)

,news_id,category
0,N55528,lifestyle
1,N19639,health
2,N61837,news
3,N53526,health
4,N38324,health
5,N2073,sports
6,N49186,weather
7,N59295,news
8,N24510,entertainment
9,N39237,news


In [6]:
clicked_records = []

for _, row in behaviors.iterrows():

    user = row["user_id"]

    for item in row["impressions"].split():

        news_id, label = item.rsplit("-", 1)

        if label == "1":
            clicked_records.append(
                [user, news_id]
            )

clicked_df = pd.DataFrame(
    clicked_records,
    columns=["user_id", "news_id"]
)

print("Total clicked article records:", len(clicked_df))

clicked_df.head()

Total clicked article records: 236344


,user_id,news_id
0,U13740,N55689
1,U91836,N17059
2,U73700,N23814
3,U34670,N49685
4,U8125,N8400


In [7]:
clicked_df = clicked_df.merge(
    news[["news_id", "category"]],
    on="news_id",
    how="left"
)

print("Missing categories:", clicked_df["category"].isna().sum())

clicked_df.head(10)

Missing categories: 0


,user_id,news_id,category
0,U13740,N55689,sports
1,U91836,N17059,finance
2,U73700,N23814,lifestyle
3,U34670,N49685,music
4,U8125,N8400,autos
5,U19739,N21119,news
6,U19739,N33619,news
7,U8355,N55204,entertainment
8,U46596,N33885,finance
9,U79199,N51048,news


In [8]:
category_counts = (
    clicked_df
    .groupby(["user_id", "category"])
    .size()
    .unstack(fill_value=0)
)

category_counts.head()

category,autos,entertainment,finance,foodanddrink,health,kids,lifestyle,movies,music,news,northamerica,sports,travel,tv,video,weather
user_id,,,,,,,,,,,,,,,,
U100,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
U1000,0,0,0,1,0,0,0,1,0,2,0,0,0,0,0,0
U10001,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0
U10003,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
U10008,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [9]:
category_shares = category_counts.div(
    category_counts.sum(axis=1),
    axis=0
)

category_shares = category_shares.add_suffix("_share")

category_shares.head()

category,autos_share,entertainment_share,finance_share,foodanddrink_share,health_share,kids_share,lifestyle_share,movies_share,music_share,news_share,northamerica_share,sports_share,travel_share,tv_share,video_share,weather_share
user_id,,,,,,,,,,,,,,,,
U100,0.000000,0.000000,0.0,0.00,0.000000,0.0,0.0,0.00,0.0,1.0,0.0,0.000000,0.000000,0.0,0.0,0.0
U1000,0.000000,0.000000,0.0,0.25,0.000000,0.0,0.0,0.25,0.0,0.5,0.0,0.000000,0.000000,0.0,0.0,0.0
U10001,0.333333,0.000000,0.0,0.00,0.000000,0.0,0.0,0.00,0.0,0.0,0.0,0.333333,0.333333,0.0,0.0,0.0
U10003,0.000000,0.333333,0.0,0.00,0.333333,0.0,0.0,0.00,0.0,0.0,0.0,0.333333,0.000000,0.0,0.0,0.0
U10008,0.000000,0.000000,0.0,0.00,0.000000,0.0,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0


In [11]:
def calculate_entropy(row):

    probabilities = row[row > 0]

    return -np.sum(
        probabilities * np.log(probabilities)
    )


content_entropy = category_shares.apply(
    calculate_entropy,
    axis=1
)

content_entropy.name = "content_entropy"

content_entropy.head()

user_id
U100     -0.000000
U1000     1.039721
U10001    1.098612
U10003    1.098612
U10008   -0.000000
Name: content_entropy, dtype: float64

In [13]:
dominant_category_share = category_shares.max(axis=1)

dominant_category_share.name = "dominant_category_share"
dominant_category = (
    category_shares
    .idxmax(axis=1)
    .str.replace("_share", "", regex=False)
)

dominant_category.name = "dominant_category"

In [14]:
categories_engaged = (
    category_counts > 0
).sum(axis=1)

categories_engaged.name = "categories_engaged"

In [15]:
user_features = user_features.set_index("user_id")

user_features = user_features.join(
    category_shares,
    how="left"
)

user_features = user_features.join(
    content_entropy,
    how="left"
)

user_features = user_features.join(
    dominant_category_share,
    how="left"
)

user_features = user_features.join(
    dominant_category,
    how="left"
)

user_features = user_features.join(
    categories_engaged,
    how="left"
)

user_features = user_features.reset_index()

user_features.head()

,user_id,session_count,total_impressions,total_clicks,history_length,overall_ctr,autos_share,entertainment_share,finance_share,foodanddrink_share,...,northamerica_share,sports_share,travel_share,tv_share,video_share,weather_share,content_entropy,dominant_category_share,dominant_category,categories_engaged
0,U100,1,105,1,10,0.009524,0.000000,0.000000,0.0,0.00,...,0.0,0.000000,0.000000,0.0,0.0,0.0,-0.000000,1.000000,news,1
1,U1000,3,205,4,3,0.019512,0.000000,0.000000,0.0,0.25,...,0.0,0.000000,0.000000,0.0,0.0,0.0,1.039721,0.500000,news,3
2,U10001,2,84,3,15,0.035714,0.333333,0.000000,0.0,0.00,...,0.0,0.333333,0.333333,0.0,0.0,0.0,1.098612,0.333333,autos,3
3,U10003,2,106,3,8,0.028302,0.000000,0.333333,0.0,0.00,...,0.0,0.333333,0.000000,0.0,0.0,0.0,1.098612,0.333333,entertainment,3
4,U10008,1,72,1,23,0.013889,0.000000,0.000000,0.0,0.00,...,0.0,0.000000,0.000000,0.0,0.0,1.0,-0.000000,1.000000,weather,1


In [16]:
print("Users:", len(user_features))
print("Features:", user_features.shape[1])

print("\nColumns:")
for col in user_features.columns:
    print(col)

Users: 50000
Features: 26

Columns:
user_id
session_count
total_impressions
total_clicks
history_length
overall_ctr
autos_share
entertainment_share
finance_share
foodanddrink_share
health_share
kids_share
lifestyle_share
movies_share
music_share
news_share
northamerica_share
sports_share
travel_share
tv_share
video_share
weather_share
content_entropy
dominant_category_share
dominant_category
categories_engaged


In [17]:
thresholds = [1, 2, 3, 5, 10, 15, 20]

print("USER RETENTION BY MINIMUM CLICK THRESHOLD")
print("-----------------------------------------")

for t in thresholds:
    remaining = (user_features["total_clicks"] >= t).sum()
    percentage = remaining / len(user_features) * 100

    print(
        f"Minimum {t:2d} clicks: "
        f"{remaining:5d} users ({percentage:.1f}%)"
    )

USER RETENTION BY MINIMUM CLICK THRESHOLD
-----------------------------------------
Minimum  1 clicks: 50000 users (100.0%)
Minimum  2 clicks: 37190 users (74.4%)
Minimum  3 clicks: 27678 users (55.4%)
Minimum  5 clicks: 16159 users (32.3%)
Minimum 10 clicks:  5836 users (11.7%)
Minimum 15 clicks:  2702 users (5.4%)
Minimum 20 clicks:  1364 users (2.7%)


In [18]:
# Historical clicked articles for each user
history_records = []

for _, row in behaviors.iterrows():

    user = row["user_id"]

    if pd.notna(row["history"]):

        for news_id in row["history"].split():

            history_records.append(
                [user, news_id]
            )

history_df = pd.DataFrame(
    history_records,
    columns=["user_id", "news_id"]
)

print("Historical interaction records:", len(history_df))
history_df.head()

Historical interaction records: 5107639


,user_id,news_id
0,U13740,N55189
1,U13740,N42782
2,U13740,N34694
3,U13740,N45794
4,U13740,N18445


In [19]:
history_df = history_df.drop_duplicates(
    subset=["user_id", "news_id"]
)

print(
    "Unique historical user-article interactions:",
    len(history_df)
)

Unique historical user-article interactions: 915011


In [20]:
all_clicks = pd.concat(
    [
        history_df,
        clicked_df[["user_id", "news_id"]]
    ],
    ignore_index=True
)

all_clicks = all_clicks.drop_duplicates(
    subset=["user_id", "news_id"]
)

print(
    "Total unique user-article interactions:",
    len(all_clicks)
)

Total unique user-article interactions: 1148447


In [21]:
all_clicks = all_clicks.merge(
    news[["news_id", "category"]],
    on="news_id",
    how="left"
)

print(
    "Interactions with missing category:",
    all_clicks["category"].isna().sum()
)

all_clicks.head()

Interactions with missing category: 0


,user_id,news_id,category
0,U13740,N55189,tv
1,U13740,N42782,sports
2,U13740,N34694,tv
3,U13740,N45794,news
4,U13740,N18445,sports


In [22]:
category_counts_full = (
    all_clicks
    .groupby(["user_id", "category"])
    .size()
    .unstack(fill_value=0)
)

category_shares_full = category_counts_full.div(
    category_counts_full.sum(axis=1),
    axis=0
)

category_shares_full = (
    category_shares_full
    .add_suffix("_share")
)

category_shares_full.head()

category,autos_share,entertainment_share,finance_share,foodanddrink_share,health_share,kids_share,lifestyle_share,middleeast_share,movies_share,music_share,news_share,northamerica_share,sports_share,travel_share,tv_share,video_share,weather_share
user_id,,,,,,,,,,,,,,,,,
U100,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.181818,0.0,0.000000,0.090909,0.181818,0.0,0.181818,0.000000,0.363636,0.0,0.000000
U1000,0.000000,0.142857,0.142857,0.142857,0.000000,0.0,0.142857,0.0,0.142857,0.000000,0.285714,0.0,0.000000,0.000000,0.000000,0.0,0.000000
U10001,0.055556,0.000000,0.000000,0.055556,0.000000,0.0,0.055556,0.0,0.000000,0.000000,0.166667,0.0,0.555556,0.055556,0.055556,0.0,0.000000
U10003,0.090909,0.090909,0.000000,0.000000,0.181818,0.0,0.090909,0.0,0.000000,0.000000,0.363636,0.0,0.181818,0.000000,0.000000,0.0,0.000000
U10008,0.000000,0.083333,0.000000,0.041667,0.041667,0.0,0.000000,0.0,0.000000,0.000000,0.458333,0.0,0.041667,0.041667,0.125000,0.0,0.166667


In [24]:
total_articles_consumed = (
    all_clicks
    .groupby("user_id")
    .size()
)

total_articles_consumed.describe()

count    50000.000000
mean        22.968940
std         27.829806
min          1.000000
25%          8.000000
50%         14.000000
75%         27.000000
max        658.000000
dtype: float64

In [25]:
articles_consumed = (
    all_clicks
    .groupby("user_id")
    .size()
    .rename("total_articles_consumed")
)

user_features = (
    user_features
    .set_index("user_id")
    .join(articles_consumed, how="left")
    .reset_index()
)

user_features["total_articles_consumed"] = (
    user_features["total_articles_consumed"]
    .fillna(0)
)

In [26]:
thresholds = [5, 10, 15, 20, 30]

for t in thresholds:
    n = (user_features["total_articles_consumed"] >= t).sum()
    
    print(
        f"Minimum {t:2d} consumed articles: "
        f"{n:5d} users ({n/len(user_features)*100:.1f}%)"
    )

Minimum  5 consumed articles: 46986 users (94.0%)
Minimum 10 consumed articles: 32820 users (65.6%)
Minimum 15 consumed articles: 23732 users (47.5%)
Minimum 20 consumed articles: 18031 users (36.1%)
Minimum 30 consumed articles: 11235 users (22.5%)


In [27]:
old_share_columns = [
    col for col in user_features.columns
    if col.endswith("_share")
]

# dominant_category_share also ends with "_share",
# so we'll recreate it below.

user_features = user_features.drop(
    columns=old_share_columns
)

user_features = user_features.drop(
    columns=[
        "content_entropy",
        "dominant_category",
        "categories_engaged"
    ],
    errors="ignore"
)
user_features = (
    user_features
    .set_index("user_id")
    .join(category_shares_full, how="left")
    .reset_index()
)

In [28]:
share_columns = [
    col for col in user_features.columns
    if col.endswith("_share")
]

def calculate_entropy(row):
    probabilities = row[row > 0]
    return -np.sum(
        probabilities * np.log(probabilities)
    )


user_features["content_entropy"] = (
    user_features[share_columns]
    .apply(calculate_entropy, axis=1)
)

user_features["dominant_category_share"] = (
    user_features[share_columns]
    .max(axis=1)
)

user_features["dominant_category"] = (
    user_features[share_columns]
    .idxmax(axis=1)
    .str.replace("_share", "", regex=False)
)

user_features["categories_engaged"] = (
    (user_features[share_columns] > 0)
    .sum(axis=1)
)

In [29]:
print(
    "Category shares sum:"
)

print(
    user_features[share_columns]
    .sum(axis=1)
    .describe()
)

print(
    "\nContent entropy:"
)

print(
    user_features["content_entropy"]
    .describe()
)

print(
    "\nCategories engaged:"
)

print(
    user_features["categories_engaged"]
    .describe()
)

Category shares sum:
count    5.000000e+04
mean     1.000000e+00
std      7.451979e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

Content entropy:
count    50000.000000
mean         1.517884
std          0.439465
min         -0.000000
25%          1.264867
50%          1.560710
75%          1.846220
max          2.544967
Name: content_entropy, dtype: float64

Categories engaged:
count    50000.000000
mean         6.521960
std          2.896261
min          1.000000
25%          4.000000
50%          6.000000
75%          9.000000
max         15.000000
Name: categories_engaged, dtype: float64


In [30]:
MIN_ARTICLES = 10

ml_users = user_features[
    user_features["total_articles_consumed"] >= MIN_ARTICLES
].copy()

print("Original users:", len(user_features))
print("Users retained:", len(ml_users))
print(
    "Retention:",
    round(len(ml_users) / len(user_features) * 100, 2),
    "%"
)

Original users: 50000
Users retained: 32820
Retention: 65.64 %


In [31]:
ml_users.to_csv(
    "C:\\Users\\admin\\ML_RESUME_PROJECT\\user_features.csv",
    index=False
)

print("Saved:", ml_users.shape)

Saved: (32820, 28)
